In [27]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import warnings
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [28]:
# ============================================================
# 0. データの読み込みと初期定義
# ============================================================
print("データを読み込んでいます...")
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit_template = pd.read_csv('data/sample_submit.csv', header=None)

# ベイスギのみ除外
train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns if c not in ['sample number', 'species number', '樹種', '含水率']]

# ターゲット変換は log1p
y_train_log = np.log1p(train['含水率'])
y_true = np.expm1(y_train_log)
groups = train['species number']

# 波長インデックス
wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1450 = np.argmin(np.abs(wavelengths - 1450))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

# 物理バンドマスク
scatter_band   = (wavelengths >= 1100) & (wavelengths <= 1250)
pure_dens_band = (wavelengths <= 1100)
skeleton_band  = (wavelengths >= 10000000 / 4320) & (wavelengths <= 10000000 / 4230)
water_actual   = (wavelengths >= 10000000 / 4950) & (wavelengths <= 10000000 / 4800)
lignin_band    = (wavelengths >= 10000000 / 6000) & (wavelengths <= 10000000 / 5500)
water_band1    = (wavelengths >= 1350) & (wavelengths <= 1600)
water_band2    = (wavelengths >= 1800) & (wavelengths <= 2100)
water_weak     = (wavelengths >= 1400) & (wavelengths <= 1520)
water_strong   = (wavelengths >= 1880) & (wavelengths <= 1980)
cellulose_band = (wavelengths >= 2050) & (wavelengths <= 2200)

N_PLS_COMPONENTS = 5
N_NEIGHBORS = 9

X_train_raw = train[spec_cols].values
X_test_raw  = test[spec_cols].values

print(f"train: {train.shape}, test: {test.shape}")
print(f"樹種: {sorted(train['樹種'].unique())}")


データを読み込んでいます...
train: (1210, 1559), test: (550, 1558)
樹種: ['イチョウ', 'ウエンジ', 'ウォールナット', 'クリ', 'スプルース', 'チェリー', 'トチ', 'ナラ', 'ヒノキ', 'ベイマツ', 'ホワイトオーク', '米ヒバ']


In [29]:
# ============================================================
# 1. 共通ユーティリティ関数
# ============================================================
def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


def mixup_cross_species(X, y, species, n_augment=500, alpha=0.3, seed=42):
    """異なる樹種間で Mixup を行う。
    ※ 必ず訓練フォールドのデータ (X_tr, y_tr) のみを渡すこと（リーク防止）。
    """
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[idx1] + (1 - lam) * X[idx2])
        y_aug.append(lam * y[idx1] + (1 - lam) * y[idx2])
    return np.array(X_aug), np.array(y_aug)


def mixup_within_species(X, y, species, n_augment=200, alpha=0.3, seed=42):
    """同一樹種内で Mixup を行う。
    含水率の異なるサンプル同士を補間し、樹種内の含水率変化を学習させる。
    ※ 必ず訓練フォールドのデータ (X_tr, y_tr) のみを渡すこと（リーク防止）。
    """
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        # サンプル数が 2 以上の樹種からランダムに選ぶ
        valid = [sp for sp in unique_species if np.sum(species == sp) >= 2]
        if len(valid) == 0:
            continue
        sp = rng.choice(valid)
        sp_idx = np.where(species == sp)[0]
        idx1, idx2 = rng.choice(sp_idx, size=2, replace=False)
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[idx1] + (1 - lam) * X[idx2])
        y_aug.append(lam * y[idx1] + (1 - lam) * y[idx2])
    return np.array(X_aug), np.array(y_aug)


def mixup_hybrid(X, y, species, n_cross=500, n_within=200, alpha=0.3, seed=42):
    """cross-species Mixup と within-species Mixup を組み合わせる。

    引数:
        n_cross  : 異種間の生成サンプル数（デフォルト 500）
        n_within : 同種内の生成サンプル数（デフォルト 200）

    リーク防止の設計:
        - 本関数は GroupKFold で分割した後、訓練フォールド (X_tr, y_tr) のみで呼ぶ
        - 検証フォールド・テストデータは一切使用しない
        - seed を fold 番号でずらすことで、フォールドごとに異なる拡張データを生成
    """
    X_cross,  y_cross  = mixup_cross_species(
        X, y, species, n_cross,  alpha, seed)
    X_within, y_within = mixup_within_species(
        X, y, species, n_within, alpha, seed + 1000)

    X_aug = np.vstack([X_cross, X_within])
    y_aug = np.concatenate([y_cross, y_within])
    return X_aug, y_aug


In [30]:
# ============================================================
# 2. 特徴量セット A・B・C の生成関数
# ============================================================
#
# リーク防止の原則（全関数共通）:
#   - PLS・PCA・KNN は必ず訓練フォールド (X_tr) のみで fit する
#   - val・test には transform のみ適用する
#   - ターゲット (y_tr) を使う処理は訓練フォールド内に閉じる
#
# 定数
N_PLS = 5    # 特徴量A: PLS 成分数
N_PCA = 20   # 特徴量C: PCA 成分数


# ── 物理特徴量の計算（特徴量B の内部で使用） ────────────────────────────
def extract_trap_features(X_raw):
    feats = {}
    sc_mean      = np.mean(X_raw[:, scatter_band], axis=1)
    pure_density = np.mean(X_raw[:, pure_dens_band], axis=1)
    feats['pure_density'] = pure_density

    w1_mean  = np.mean(X_raw[:, water_band1], axis=1)
    w2_mean  = np.mean(X_raw[:, water_band2], axis=1)
    ww_mean  = np.mean(X_raw[:, water_weak], axis=1)
    ws_mean  = np.mean(X_raw[:, water_strong], axis=1)
    cel_mean = np.mean(X_raw[:, cellulose_band], axis=1)

    skeleton_mean  = np.mean(X_raw[:, skeleton_band], axis=1)
    water_act_mean = np.mean(X_raw[:, water_actual], axis=1)
    lignin_mean    = np.mean(X_raw[:, lignin_band], axis=1)

    feats['water_vs_skeleton_ratio'] = water_act_mean / (skeleton_mean + 1e-8)
    feats['water_vs_lignin_ratio']   = water_act_mean / (lignin_mean + 1e-8)
    feats['w2_div_scatter']          = w2_mean / (sc_mean + 1e-8)
    feats['w2_div_pure_density']     = w2_mean / (pure_density + 1e-8)
    feats['ws_div_pure_density']     = ws_mean / (pure_density + 1e-8)

    water2_region  = X_raw[:, water_band2]
    peak_idx_local = np.argmax(water2_region, axis=1)
    feats['peak_wl_w2'] = wavelengths[water_band2][peak_idx_local]

    feats['abs1450_div_1940'] = X_raw[:, idx_1450] / (X_raw[:, idx_1940] + 1e-8)
    feats['weak_div_strong']  = ww_mean / (ws_mean + 1e-8)
    feats['cel_div_w2']       = cel_mean / (w2_mean + 1e-8)

    feats['scatter_mean'] = sc_mean
    feats['scatter_std']  = np.std(X_raw[:, scatter_band], axis=1)
    feats['raw_mean']     = np.mean(X_raw, axis=1)

    snv = apply_snv(X_raw)
    d1  = savgol_filter(snv, 15, 2, deriv=1, axis=1)
    feats['d1_w2_max']   = np.max(d1[:, water_band2], axis=1)
    feats['d1_w2_min']   = np.min(d1[:, water_band2], axis=1)
    feats['d1_w2_range'] = feats['d1_w2_max'] - feats['d1_w2_min']

    d2 = savgol_filter(snv, 15, 2, deriv=2, axis=1)
    feats['d2_at1940'] = d2[:, idx_1940]
    feats['d2_at1450'] = d2[:, idx_1450]

    d2_w2_max = np.max(d2[:, water_band2], axis=1)
    d2_w2_min = np.min(d2[:, water_band2], axis=1)
    feats['d2_w2_curvature_ratio'] = d2_w2_max / (np.abs(d2_w2_min) + 1e-8)

    return pd.DataFrame(feats).values, list(feats.keys())


# ── 特徴量A: PLSスコア + KNN距離加重予測値 ──────────────────────────────
def get_feature_A(X_tr, y_tr, X_aug, X_va, X_te, n_orig):
    """
    含水率と相関の強い方向に次元圧縮（PLS）し、
    近傍サンプルの含水率を距離加重で参照（KNN）する。

    リーク防止:
        PLS・KNN ともに X_tr のみで fit し、
        aug・val・test には transform/predict のみ適用する。
    """
    snv_tr  = apply_snv(X_tr)
    snv_aug = apply_snv(X_aug)
    snv_va  = apply_snv(X_va)
    snv_te  = apply_snv(X_te)

    # PLS: X_tr のみで fit
    pls = PLSRegression(n_components=N_PLS, scale=False)
    pls.fit(snv_tr, y_tr)
    score_tr  = pls.transform(snv_tr)
    score_aug = pls.transform(snv_aug)
    score_va  = pls.transform(snv_va)
    score_te  = pls.transform(snv_te)

    # KNN: X_tr のスコアのみで fit
    knn = NearestNeighbors(n_neighbors=N_NEIGHBORS, metric='cosine')
    knn.fit(score_tr)

    def _knn_pred(query_scores, is_aug=False):
        n_ask = N_NEIGHBORS + 1 if is_aug else N_NEIGHBORS
        dist, idx = knn.kneighbors(query_scores, n_neighbors=n_ask)
        preds = []
        for i in range(len(query_scores)):
            nb = idx[i]; d = dist[i]
            if is_aug:
                mask = nb != i if i < n_orig else np.ones(len(nb), dtype=bool)
                nb = nb[mask][:N_NEIGHBORS]
                d  = d[mask][:N_NEIGHBORS]
            preds.append(np.average(y_tr[:n_orig][nb], weights=1.0 / (d + 1e-5)))
        return np.array(preds).reshape(-1, 1)

    knn_aug = _knn_pred(score_aug, is_aug=True)
    knn_va  = _knn_pred(score_va)
    knn_te  = _knn_pred(score_te)

    F_aug = np.hstack([score_aug, knn_aug])
    F_va  = np.hstack([score_va,  knn_va])
    F_te  = np.hstack([score_te,  knn_te])
    return F_aug, F_va, F_te


# ── 特徴量B: 物理ベース特徴量のみ ────────────────────────────────────────
def get_feature_B(X_aug, X_va, X_te):
    """
    水の物理状態を直接反映した比率・形状特徴量。
    ターゲット（含水率）を一切使わないため fit 不要。リーク原理的に不可能。
    """
    F_aug, _ = extract_trap_features(X_aug)
    F_va,  _ = extract_trap_features(X_va)
    F_te,  _ = extract_trap_features(X_te)
    return F_aug, F_va, F_te


# ── 特徴量C: SNV + SG1次微分 + PCA圧縮 ──────────────────────────────────
def get_feature_C(X_tr, X_aug, X_va, X_te):
    """
    生スペクトル情報を最大限活用する。SNV + 1次微分を結合して PCA で圧縮。
    SVR・Ridge など高次元が苦手なモデルにも対応できる形にする。

    リーク防止:
        PCA は X_tr のみで fit し、aug・val・test には transform のみ適用する。
    """
    snv_tr  = apply_snv(X_tr)
    snv_aug = apply_snv(X_aug)
    snv_va  = apply_snv(X_va)
    snv_te  = apply_snv(X_te)

    d1_tr  = savgol_filter(snv_tr,  15, 2, deriv=1, axis=1)
    d1_aug = savgol_filter(snv_aug, 15, 2, deriv=1, axis=1)
    d1_va  = savgol_filter(snv_va,  15, 2, deriv=1, axis=1)
    d1_te  = savgol_filter(snv_te,  15, 2, deriv=1, axis=1)

    # SNV + 1次微分を結合（約 3110 次元）
    comb_tr  = np.hstack([snv_tr,  d1_tr])
    comb_aug = np.hstack([snv_aug, d1_aug])
    comb_va  = np.hstack([snv_va,  d1_va])
    comb_te  = np.hstack([snv_te,  d1_te])

    # PCA: X_tr のみで fit
    pca = PCA(n_components=N_PCA, random_state=42)
    pca.fit(comb_tr)

    F_aug = pca.transform(comb_aug)
    F_va  = pca.transform(comb_va)
    F_te  = pca.transform(comb_te)
    return F_aug, F_va, F_te


print("特徴量セット A・B・C の関数を定義しました")
print(f"  特徴量A の次元: PLS {N_PLS}成分 + KNN 1 = {N_PLS + 1}")
print(f"  特徴量B の次元: 物理特徴量 {len(extract_trap_features(X_train_raw[:1])[1])} 個")
print(f"  特徴量C の次元: PCA {N_PCA}成分")


特徴量セット A・B・C の関数を定義しました
  特徴量A の次元: PLS 5成分 + KNN 1 = 6
  特徴量B の次元: 物理特徴量 19 個
  特徴量C の次元: PCA 20成分


In [31]:
# ============================================================
# 3. 特徴量ディスパッチ + LightGBM ランナー
# ============================================================

def _get_features(feat_set, X_tr, y_tr, X_aug, X_va, X_te, n_orig):
    """特徴量セット A/B/C へのディスパッチ関数。"""
    if feat_set == 'A':
        return get_feature_A(X_tr, y_tr, X_aug, X_va, X_te, n_orig)
    elif feat_set == 'B':
        return get_feature_B(X_aug, X_va, X_te)
    elif feat_set == 'C':
        return get_feature_C(X_tr, X_aug, X_va, X_te)
    else:
        raise ValueError(f"Unknown feat_set: {feat_set}")


DEFAULT_LGB_PARAMS = {
    'n_estimators'    : 1500,
    'learning_rate'   : 0.02,
    'max_depth'       : 4,
    'num_leaves'      : 15,
    'subsample'       : 0.8,
    'colsample_bytree': 0.6,
    'min_child_samples': 20,
    'reg_alpha'       : 0.2,
    'reg_lambda'      : 3.0,
}


def run_lgb(feat_set, params=None, seed=42):
    """LightGBM ランナー（特徴量セット A/B/C に対応）。

    リーク防止:
        - Mixup は GroupKFold 分割後の訓練フォールド (X_tr) のみで生成
        - 特徴量の fit（PLS・PCA・KNN）はすべて X_tr のみで実施
        - val・test には transform のみ適用
    """
    if params is None:
        params = DEFAULT_LGB_PARAMS

    gkf = GroupKFold(n_splits=5)
    oof_pred   = np.zeros(len(train))
    final_pred = np.zeros(len(test))

    for fold, (tr_idx, va_idx) in enumerate(
            gkf.split(X_train_raw, y_train_log, groups)):

        X_tr  = X_train_raw[tr_idx]
        y_tr  = y_train_log.iloc[tr_idx].values
        X_va  = X_train_raw[va_idx]
        y_va  = y_train_log.iloc[va_idx].values
        tr_sp = groups.iloc[tr_idx].values

        # Mixup: 訓練フォールドのみ使用（リーク防止）
        X_mix, y_mix = mixup_hybrid(
            X_tr, y_tr, tr_sp, seed=seed + fold)
        n_orig = len(X_tr)
        X_aug  = np.vstack([X_tr, X_mix])
        y_aug  = np.concatenate([y_tr, y_mix])

        # 特徴量取得（内部で fit は X_tr のみ）
        F_aug, F_va, F_te = _get_features(
            feat_set, X_tr, y_tr, X_aug, X_va, X_test_raw, n_orig)

        model = lgb.LGBMRegressor(
            **params, random_state=seed, verbosity=-1)
        model.fit(F_aug, y_aug,
                  eval_set=[(F_va, y_va)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        oof_pred[va_idx] = np.expm1(model.predict(F_va))
        final_pred      += np.expm1(model.predict(F_te)) / 5

    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    return {'oof_pred': oof_pred, 'test_pred': final_pred, 'oof_rmse': oof_rmse}


In [32]:
# ============================================================
# 4. SVR ランナー（RBF カーネル）
# ============================================================

DEFAULT_SVR_PARAMS = {
    'kernel' : 'rbf',
    'C'      : 10.0,
    'epsilon': 0.1,
    'gamma'  : 'scale',
}


def run_svr(feat_set, params=None, seed=42):
    """SVR ランナー（特徴量セット A/B/C に対応）。

    リーク防止:
        - Mixup・特徴量 fit は run_lgb と同じ設計
        - StandardScaler は X_tr 由来の特徴量（F_aug[:n_orig]）のみで fit
          → val・test には transform のみ適用
    """
    if params is None:
        params = DEFAULT_SVR_PARAMS

    gkf = GroupKFold(n_splits=5)
    oof_pred   = np.zeros(len(train))
    final_pred = np.zeros(len(test))

    for fold, (tr_idx, va_idx) in enumerate(
            gkf.split(X_train_raw, y_train_log, groups)):

        X_tr  = X_train_raw[tr_idx]
        y_tr  = y_train_log.iloc[tr_idx].values
        X_va  = X_train_raw[va_idx]
        y_va  = y_train_log.iloc[va_idx].values
        tr_sp = groups.iloc[tr_idx].values

        # Mixup: 訓練フォールドのみ（リーク防止）
        X_mix, y_mix = mixup_hybrid(
            X_tr, y_tr, tr_sp, seed=seed + fold)
        n_orig = len(X_tr)
        X_aug  = np.vstack([X_tr, X_mix])
        y_aug  = np.concatenate([y_tr, y_mix])

        # 特徴量取得（内部 fit は X_tr のみ）
        F_aug, F_va, F_te = _get_features(
            feat_set, X_tr, y_tr, X_aug, X_va, X_test_raw, n_orig)

        # スケーリング: X_tr 由来の特徴量のみで fit（リーク防止）
        scaler = StandardScaler()
        scaler.fit(F_aug[:n_orig])
        F_aug_s = scaler.transform(F_aug)
        F_va_s  = scaler.transform(F_va)
        F_te_s  = scaler.transform(F_te)

        model = SVR(**params)
        model.fit(F_aug_s, y_aug)

        oof_pred[va_idx] = np.expm1(model.predict(F_va_s))
        final_pred      += np.expm1(model.predict(F_te_s)) / 5

    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    return {'oof_pred': oof_pred, 'test_pred': final_pred, 'oof_rmse': oof_rmse}


In [33]:
# ============================================================
# 5. Ridge Regression ランナー
# ============================================================

DEFAULT_RIDGE_PARAMS = {
    'alpha': 1.0,
}


def run_ridge_model(feat_set, params=None):
    """Ridge Regression ランナー（特徴量セット A/B/C に対応）。

    リーク防止:
        - Mixup・特徴量 fit は run_lgb と同じ設計
        - StandardScaler は F_aug[:n_orig] のみで fit
    """
    if params is None:
        params = DEFAULT_RIDGE_PARAMS

    gkf = GroupKFold(n_splits=5)
    oof_pred   = np.zeros(len(train))
    final_pred = np.zeros(len(test))

    for fold, (tr_idx, va_idx) in enumerate(
            gkf.split(X_train_raw, y_train_log, groups)):

        X_tr  = X_train_raw[tr_idx]
        y_tr  = y_train_log.iloc[tr_idx].values
        X_va  = X_train_raw[va_idx]
        y_va  = y_train_log.iloc[va_idx].values
        tr_sp = groups.iloc[tr_idx].values

        # Mixup: 訓練フォールドのみ（リーク防止）
        X_mix, y_mix = mixup_hybrid(X_tr, y_tr, tr_sp, seed=42 + fold)
        n_orig = len(X_tr)
        X_aug  = np.vstack([X_tr, X_mix])
        y_aug  = np.concatenate([y_tr, y_mix])

        # 特徴量取得（内部 fit は X_tr のみ）
        F_aug, F_va, F_te = _get_features(
            feat_set, X_tr, y_tr, X_aug, X_va, X_test_raw, n_orig)

        # スケーリング: X_tr 由来の特徴量のみで fit（リーク防止）
        scaler = StandardScaler()
        scaler.fit(F_aug[:n_orig])
        F_aug_s = scaler.transform(F_aug)
        F_va_s  = scaler.transform(F_va)
        F_te_s  = scaler.transform(F_te)

        model = Ridge(**params)
        model.fit(F_aug_s, y_aug)

        oof_pred[va_idx] = np.expm1(model.predict(F_va_s))
        final_pred      += np.expm1(model.predict(F_te_s)) / 5

    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    return {'oof_pred': oof_pred, 'test_pred': final_pred, 'oof_rmse': oof_rmse}


In [34]:
# ============================================================
# 6. Optuna ハイパーパラメータチューニング（9モデル独立）
# ============================================================
# 各 (アルゴリズム × 特徴量) の組み合わせごとに独立して最適化
# リーク防止: objective 内で GroupKFold を毎回実行、test は不使用

def _oof_rmse(oof_log):
    return np.sqrt(mean_squared_error(y_true, np.expm1(oof_log)))


# ── LightGBM objective factory ───────────────────────────────
def make_objective_lgb(feat_set):
    def objective(trial):
        params = {
            'n_estimators'    : 1500,
            'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth'       : trial.suggest_int('max_depth', 3, 6),
            'num_leaves'      : trial.suggest_int('num_leaves', 10, 50),
            'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
            'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True),
            'reg_lambda'      : trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        }
        gkf = GroupKFold(n_splits=5)
        oof = np.zeros(len(train))
        for fold, (tr_idx, va_idx) in enumerate(
                gkf.split(X_train_raw, y_train_log, groups)):
            X_tr  = X_train_raw[tr_idx]
            y_tr  = y_train_log.iloc[tr_idx].values
            X_va  = X_train_raw[va_idx]
            y_va  = y_train_log.iloc[va_idx].values
            tr_sp = groups.iloc[tr_idx].values
            X_mix, y_mix = mixup_hybrid(X_tr, y_tr, tr_sp, seed=42 + fold)
            n_orig = len(X_tr)
            X_aug  = np.vstack([X_tr, X_mix])
            y_aug  = np.concatenate([y_tr, y_mix])
            F_aug, F_va, _ = _get_features(
                feat_set, X_tr, y_tr, X_aug, X_va, X_test_raw, n_orig)
            model = lgb.LGBMRegressor(**params, random_state=42, verbosity=-1)
            model.fit(F_aug, y_aug,
                      eval_set=[(F_va, y_va)],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            oof[va_idx] = model.predict(F_va)
        return _oof_rmse(oof)
    return objective


# ── SVR objective factory ────────────────────────────────────
def make_objective_svr(feat_set):
    def objective(trial):
        params = {
            'kernel' : 'rbf',
            'C'      : trial.suggest_float('C', 0.1, 100.0, log=True),
            'epsilon': trial.suggest_float('epsilon', 0.01, 1.0, log=True),
            'gamma'  : trial.suggest_categorical('gamma', ['scale', 'auto']),
        }
        gkf = GroupKFold(n_splits=5)
        oof = np.zeros(len(train))
        for fold, (tr_idx, va_idx) in enumerate(
                gkf.split(X_train_raw, y_train_log, groups)):
            X_tr  = X_train_raw[tr_idx]
            y_tr  = y_train_log.iloc[tr_idx].values
            X_va  = X_train_raw[va_idx]
            tr_sp = groups.iloc[tr_idx].values
            X_mix, y_mix = mixup_hybrid(X_tr, y_tr, tr_sp, seed=42 + fold)
            n_orig = len(X_tr)
            X_aug  = np.vstack([X_tr, X_mix])
            y_aug  = np.concatenate([y_tr, y_mix])
            F_aug, F_va, _ = _get_features(
                feat_set, X_tr, y_tr, X_aug, X_va, X_test_raw, n_orig)
            scaler = StandardScaler()
            scaler.fit(F_aug[:n_orig])
            model = SVR(**params)
            model.fit(scaler.transform(F_aug), y_aug)
            oof[va_idx] = model.predict(scaler.transform(F_va))
        return _oof_rmse(oof)
    return objective


# ── Ridge objective factory ──────────────────────────────────
def make_objective_ridge(feat_set):
    def objective(trial):
        params = {
            'alpha': trial.suggest_float('alpha', 1e-3, 100.0, log=True),
        }
        gkf = GroupKFold(n_splits=5)
        oof = np.zeros(len(train))
        for fold, (tr_idx, va_idx) in enumerate(
                gkf.split(X_train_raw, y_train_log, groups)):
            X_tr  = X_train_raw[tr_idx]
            y_tr  = y_train_log.iloc[tr_idx].values
            X_va  = X_train_raw[va_idx]
            tr_sp = groups.iloc[tr_idx].values
            X_mix, y_mix = mixup_hybrid(X_tr, y_tr, tr_sp, seed=42 + fold)
            n_orig = len(X_tr)
            X_aug  = np.vstack([X_tr, X_mix])
            y_aug  = np.concatenate([y_tr, y_mix])
            F_aug, F_va, _ = _get_features(
                feat_set, X_tr, y_tr, X_aug, X_va, X_test_raw, n_orig)
            scaler = StandardScaler()
            scaler.fit(F_aug[:n_orig])
            model = Ridge(**params)
            model.fit(scaler.transform(F_aug), y_aug)
            oof[va_idx] = model.predict(scaler.transform(F_va))
        return _oof_rmse(oof)
    return objective


# ── 9モデル独立チューニング ──────────────────────────────────
MODEL_CONFIGS = [
    ('lgb',   'A', make_objective_lgb,   50),
    ('lgb',   'B', make_objective_lgb,   50),
    ('lgb',   'C', make_objective_lgb,   50),
    ('svr',   'A', make_objective_svr,   30),
    ('svr',   'B', make_objective_svr,   30),
    ('svr',   'C', make_objective_svr,   30),
    ('ridge', 'A', make_objective_ridge, 20),
    ('ridge', 'B', make_objective_ridge, 20),
    ('ridge', 'C', make_objective_ridge, 20),
]

DEFAULT_PARAMS = {
    'lgb'  : DEFAULT_LGB_PARAMS,
    'svr'  : DEFAULT_SVR_PARAMS,
    'ridge': DEFAULT_RIDGE_PARAMS,
}

best_params = {}

print("Optuna チューニング開始（9モデル独立）...")
print("=" * 60)

for algo, feat, make_fn, n_trials in MODEL_CONFIGS:
    key = f'{algo}_{feat}'
    print(f"\n[{key}] {n_trials} trials ...")
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_fn(feat), n_trials=n_trials, show_progress_bar=True)
    best_params[key] = {**DEFAULT_PARAMS[algo], **study.best_params}
    print(f"  best OOF RMSE : {study.best_value:.4f}")
    print(f"  best params   : {study.best_params}")

print("\n" + "=" * 60)
print("チューニング完了")


Optuna チューニング開始（9モデル独立）...

[lgb_A] 50 trials ...


Best trial: 1. Best value: 18.7028:  14%|█▍        | 7/50 [00:20<02:04,  2.89s/it]


[W 2026-06-29 20:51:40,252] Trial 7 failed with parameters: {'learning_rate': 0.012260057359187526, 'max_depth': 3, 'num_leaves': 11, 'subsample': 0.6626651653816322, 'colsample_bytree': 0.5720741027826374, 'min_child_samples': 21, 'reg_alpha': 1.1627201204016835, 'reg_lambda': 0.5170191786366992} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\keisu\Desktop\Competition\Signate\Signate_Spectrum\.venv\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\keisu\AppData\Local\Temp\ipykernel_5208\905570798.py", line 38, in objective
    F_aug, F_va, _ = _get_features(
  File "C:\Users\keisu\AppData\Local\Temp\ipykernel_5208\3978017715.py", line 8, in _get_features
    return get_feature_A(X_tr, y_tr, X_aug, X_va, X_te, n_orig)
  File "C:\Users\keisu\AppData\Local\Temp\ipykernel_5208\3355117755.py", line 84, in get_feature_A
    pls.fit(snv_tr, y_tr)
  File "c:\Users\k

KeyboardInterrupt: 

In [ ]:
# ============================================================
# 7. 全9モデルの実行（モデルごとの最適パラメータを使用）
# ============================================================
print("=" * 60)
print("全9モデルを実行します")
print("=" * 60)

results = {}

print("\n[LightGBM]")
for feat in ['A', 'B', 'C']:
    key = f'lgb_{feat}'
    print(f"  {key} ...", end=" ", flush=True)
    results[key] = run_lgb(feat, params=best_params[key], seed=42)
    print(f"OOF RMSE = {results[key]['oof_rmse']:.4f}")

print("\n[SVR]")
for feat in ['A', 'B', 'C']:
    key = f'svr_{feat}'
    print(f"  {key} ...", end=" ", flush=True)
    results[key] = run_svr(feat, params=best_params[key], seed=42)
    print(f"OOF RMSE = {results[key]['oof_rmse']:.4f}")

print("\n[Ridge]")
for feat in ['A', 'B', 'C']:
    key = f'ridge_{feat}'
    print(f"  {key} ...", end=" ", flush=True)
    results[key] = run_ridge_model(feat, params=best_params[key])
    print(f"OOF RMSE = {results[key]['oof_rmse']:.4f}")

print("\n" + "=" * 60)
print("OOF RMSE 一覧")
print("=" * 60)
for k in sorted(results, key=lambda x: results[x]['oof_rmse']):
    print(f"  {k:<12s}  {results[k]['oof_rmse']:.4f}")


In [ ]:
# ============================================================
# 8. スタッキング（Ridge メタ学習器）
# ============================================================
# Level 0: 9モデルの OOF 予測（各サンプルに1回ずつの予測）
# Level 1: Ridge がその9列を入力して最終予測を出す
#
# リーク防止:
#   OOF 予測はすでにリークフリー（各サンプルを見ていない
#   モデルが予測した値）なので、そのまま Ridge に投入して良い

model_keys = [f'{algo}_{feat}'
              for algo in ['lgb', 'svr', 'ridge']
              for feat in ['A', 'B', 'C']]

# Level 0 の出力を行列に変換
oof_matrix  = np.column_stack([results[k]['oof_pred']  for k in model_keys])
test_matrix = np.column_stack([results[k]['test_pred'] for k in model_keys])
# shape: (n_train, 9), (n_test, 9)

# メタ特徴量のスケーリング（Ridge は距離に敏感）
meta_scaler = StandardScaler()
oof_s  = meta_scaler.fit_transform(oof_matrix)
test_s = meta_scaler.transform(test_matrix)

# Ridge メタ学習器を OOF で学習
meta_model = Ridge(alpha=1.0)
meta_model.fit(oof_s, y_true)

# テスト予測
stacking_pred = meta_model.predict(test_s)
stacking_pred = np.clip(stacking_pred, 0, None)

# OOF での擬似スコア確認（参考値）
oof_stacking = meta_model.predict(oof_s)
oof_stacking = np.clip(oof_stacking, 0, None)
oof_rmse_stack = np.sqrt(mean_squared_error(y_true, oof_stacking))

print("=" * 60)
print("スタッキング結果")
print("=" * 60)
print(f"  メタ学習器: Ridge (alpha=1.0)")
print(f"  入力モデル数: {len(model_keys)}")
print(f"  OOF RMSE（参考）: {oof_rmse_stack:.4f}")
print()
print("  各モデルの重み（係数）:")
for k, coef in zip(model_keys, meta_model.coef_):
    print(f"    {k:<12s}  {coef:+.4f}")


スタッキング結果
  メタ学習器: Ridge (alpha=1.0)
  入力モデル数: 9
  OOF RMSE（参考）: 13.7946

  各モデルの重み（係数）:
    lgb_A         +19.8842
    lgb_B         +27.2894
    lgb_C         +8.0272
    svr_A         -10.4285
    svr_B         -1.7477
    svr_C         -1.0909
    ridge_A       +2.1955
    ridge_B       -0.3553
    ridge_C       -4.2860


In [ ]:
# ============================================================
# 9. 提出ファイルの出力
# ============================================================
output_df = submit_template.copy()
output_df[1] = stacking_pred

output_filename = "koyama_stacking_submission.csv"
output_df.to_csv(output_filename, index=False, header=False)

print(f"出力ファイル: {output_filename}")
print(f"予測値の範囲: {stacking_pred.min():.2f} 〜 {stacking_pred.max():.2f}")
print(f"予測値の平均: {stacking_pred.mean():.2f}")


出力ファイル: koyama_stacking_submission.csv
予測値の範囲: 1.70 〜 165.52
予測値の平均: 37.70
